# Load Data Validation — PJM RTO

**Sources:**
- `{schema}.staging_v1_pjm_load_rt_metered_hourly` via `load_rt_metered_hourly.pull()` — RT metered load (2014+)
- `{schema}.staging_v1_pjm_load_da_hourly` via `load_da_hourly.pull()` — DA load forecast (2020+)

**Region:** RTO  
**Purpose:** Validate raw hourly load data before it enters the like-day feature pipeline.

### Checks
1. **Setup & Data Pull** — Pull RT metered and DA load, print shape/date range/columns
2. **Completeness** — Hours per day (flag != 24), date gaps, duplicates
3. **Nulls & Outliers** — Null counts, descriptive stats, z-score outliers, physical bounds
4. **Distributions** — Histograms for RT and DA load, monthly boxplot
5. **Load Time Series** — Daily avg RT load, DA overlay, day-over-day change
6. **DA vs RT Comparison** — Scatter, correlation, monthly difference boxplot
7. **Diurnal Profile** — Avg hourly load shape by season, weekday vs weekend
8. **Feature Sanity Check** — Run `load_features.build()`, correlation heatmap
9. **Recent Spot Check** — Last 7 days summary, last 3 days hourly profiles
10. **Validation Summary** — Pass/fail checks

## 1. Setup & Data Pull

In [ ]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.like_day_forecast.settings
from src.like_day_forecast.data import load_rt_metered_hourly, load_da_hourly
from src.like_day_forecast import configs

HOURS = list(range(1, 25))

In [ ]:
# Pull RT metered load and DA load
df_rt = load_rt_metered_hourly.pull()
df_rt["date"] = pd.to_datetime(df_rt["date"])

df_da = load_da_hourly.pull()
df_da["date"] = pd.to_datetime(df_da["date"])

for label, df, col in [("RT Metered", df_rt, "rt_load_mw"), ("DA", df_da, "da_load_mw")]:
    print(f"--- {label} Load ---")
    print(f"  Shape: {df.shape}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Unique dates: {df['date'].nunique():,}")
    print(f"  Columns: {list(df.columns)}")
    print()

### Previous 3 Days — RT Metered, DA, and DA-RT Difference

Quick visual inspection of the most recent 3 days across all load components.

In [ ]:
# Previous 3 days — RT metered hourly load
last_3_rt = sorted(df_rt["date"].unique())[-3:]
df_rt_last3 = df_rt[df_rt["date"].isin(last_3_rt)].sort_values(["date", "hour_ending"])
print(f"=== RT Metered Load — Previous 3 Days ===")
print(f"Dates: {[str(d.date()) for d in last_3_rt]}")
display(df_rt_last3[["date", "hour_ending", "rt_load_mw"]])

# Previous 3 days — DA hourly load
last_3_da = sorted(df_da["date"].unique())[-3:]
df_da_last3 = df_da[df_da["date"].isin(last_3_da)].sort_values(["date", "hour_ending"])
print(f"\n=== DA Load — Previous 3 Days ===")
print(f"Dates: {[str(d.date()) for d in last_3_da]}")
display(df_da_last3[["date", "hour_ending", "da_load_mw"]])

# Previous 3 days — DA-RT difference (daily avg)
overlap_dates = sorted(set(last_3_rt) & set(last_3_da))
if overlap_dates:
    rt_daily = df_rt[df_rt["date"].isin(overlap_dates)].groupby("date")["rt_load_mw"].mean().rename("rt_daily_avg")
    da_daily = df_da[df_da["date"].isin(overlap_dates)].groupby("date")["da_load_mw"].mean().rename("da_daily_avg")
    diff_last3 = pd.concat([rt_daily, da_daily], axis=1).dropna()
    diff_last3["da_rt_diff_mw"] = diff_last3["da_daily_avg"] - diff_last3["rt_daily_avg"]
    diff_last3["da_rt_diff_pct"] = (diff_last3["da_rt_diff_mw"] / diff_last3["rt_daily_avg"] * 100).round(2)
    print(f"\n=== DA - RT Load Difference (Daily Avg) — Previous 3 Days ===")
    display(diff_last3.round(0))
else:
    print("\nNo overlapping dates for DA-RT comparison")

## 2. Completeness Checks

Verify every date has 24 hourly observations, identify date gaps, and check for duplicates. Both RT and DA separately.

In [ ]:
for label, df in [("RT Metered", df_rt), ("DA", df_da)]:
    print(f"=== {label} Load ===\n")

    # Hours per day
    hours_per_day = df.groupby("date")["hour_ending"].nunique()
    incomplete_days = hours_per_day[hours_per_day != 24]
    print(f"Total dates: {len(hours_per_day):,}")
    print(f"Dates with != 24 hours: {len(incomplete_days)}")
    if len(incomplete_days) > 0:
        print(f"  Incomplete days (showing up to 10):")
        for d, h in incomplete_days.head(10).items():
            print(f"    {d}: {h} hours")
    print()

    # Date gaps
    all_dates = pd.to_datetime(sorted(df["date"].unique()))
    expected_dates = pd.date_range(all_dates.min(), all_dates.max(), freq="D")
    missing_dates = expected_dates.difference(all_dates)
    print(f"Expected dates in range: {len(expected_dates):,}")
    print(f"Missing dates (gaps): {len(missing_dates)}")
    if len(missing_dates) > 0:
        print(f"  Missing (showing up to 10): {[d.date() for d in missing_dates[:10]]}")
    print()

    # Duplicates
    dupes = df.duplicated(subset=["date", "hour_ending"], keep=False)
    print(f"Duplicate (date, hour_ending) rows: {dupes.sum()}")
    if dupes.sum() > 0:
        print(df[dupes].head(10))
    print("\n" + "=" * 60 + "\n")

## 3. Null & Outlier Analysis

Null counts, descriptive statistics, z-score outliers, and physical bounds checks. PJM RTO load typically ranges 50,000-180,000 MW; flag values below 30,000 or above 200,000 MW.

In [ ]:
LOAD_LOW = 30_000
LOAD_HIGH = 200_000

for label, df, col in [("RT Metered", df_rt, "rt_load_mw"), ("DA", df_da, "da_load_mw")]:
    print(f"=== {label} Load ===\n")

    # Null counts
    nulls = df.isnull().sum()
    print("Null counts:")
    print(nulls.to_string())
    print()

    # Descriptive statistics
    print("Descriptive statistics:")
    print(df[col].describe().to_string())
    print()

    # Z-score outliers (|z| > 3)
    z_scores = (df[col] - df[col].mean()) / df[col].std()
    z_outliers = df[z_scores.abs() > 3]
    print(f"Z-score outliers (|z| > 3): {len(z_outliers)}")
    if len(z_outliers) > 0:
        print(f"  Range: {z_outliers[col].min():,.0f} to {z_outliers[col].max():,.0f} MW")
        print(f"  Dates (up to 5): {[d.date() for d in z_outliers['date'].head(5)]}")
    print()

    # Physical bounds
    below = df[df[col] < LOAD_LOW]
    above = df[df[col] > LOAD_HIGH]
    print(f"Below {LOAD_LOW:,} MW: {len(below)} rows")
    if len(below) > 0:
        print(below[["date", "hour_ending", col]].head(5))
    print(f"Above {LOAD_HIGH:,} MW: {len(above)} rows")
    if len(above) > 0:
        print(above[["date", "hour_ending", col]].head(5))

    print("\n" + "=" * 60 + "\n")

## 4. Distributions

Side-by-side histograms for RT and DA load. Monthly boxplot of RT load to reveal seasonal patterns.

In [ ]:
# Side-by-side histograms: RT vs DA load
fig = make_subplots(rows=1, cols=2, subplot_titles=("RT Metered Load (MW)", "DA Load (MW)"))

fig.add_trace(
    go.Histogram(x=df_rt["rt_load_mw"], nbinsx=80, marker_color="#636EFA", name="RT"),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=df_da["da_load_mw"], nbinsx=80, marker_color="#EF553B", name="DA"),
    row=1, col=2,
)

fig.update_layout(
    title="Hourly Load Distribution — RT Metered vs DA",
    height=400,
    showlegend=False,
    template="plotly_dark",
)
fig.update_xaxes(title_text="Load (MW)")
fig.update_yaxes(title_text="Count")
fig.show()

In [ ]:
# Monthly boxplot of RT load — seasonal patterns
df_rt_plot = df_rt.copy()
df_rt_plot["month"] = df_rt_plot["date"].dt.month
df_rt_plot["month_name"] = df_rt_plot["date"].dt.strftime("%b")

# Sort by month number for correct ordering
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig = px.box(
    df_rt_plot,
    x="month_name",
    y="rt_load_mw",
    category_orders={"month_name": month_order},
    title="RT Metered Load by Month — Seasonal Pattern",
    labels={"rt_load_mw": "RT Load (MW)", "month_name": "Month"},
    template="plotly_dark",
)
fig.update_layout(height=450)
fig.show()

## 5. Load Time Series

Daily average RT load over full history. Overlay DA load where available (2020+). Day-over-day change with extreme jump detection.

In [ ]:
# Daily average load
rt_daily = df_rt.groupby("date")["rt_load_mw"].mean().reset_index()
rt_daily.columns = ["date", "rt_daily_avg"]

da_daily = df_da.groupby("date")["da_load_mw"].mean().reset_index()
da_daily.columns = ["date", "da_daily_avg"]

# Time series: RT full history + DA overlay
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rt_daily["date"], y=rt_daily["rt_daily_avg"],
    mode="lines", name="RT Metered",
    line=dict(color="#636EFA", width=0.8),
))

fig.add_trace(go.Scatter(
    x=da_daily["date"], y=da_daily["da_daily_avg"],
    mode="lines", name="DA Forecast",
    line=dict(color="#EF553B", width=0.8),
))

fig.update_layout(
    title="Daily Average Load — RT Metered (2014+) & DA (2020+)",
    xaxis_title="Date",
    yaxis_title="Avg Load (MW)",
    height=450,
    template="plotly_dark",
)
fig.show()

In [ ]:
# Day-over-day change with extreme jump detection
rt_daily_sorted = rt_daily.sort_values("date").copy()
rt_daily_sorted["daily_change"] = rt_daily_sorted["rt_daily_avg"].diff()

change_std = rt_daily_sorted["daily_change"].std()
change_mean = rt_daily_sorted["daily_change"].mean()
threshold = 3 * change_std

extreme_jumps = rt_daily_sorted[rt_daily_sorted["daily_change"].abs() > threshold]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rt_daily_sorted["date"], y=rt_daily_sorted["daily_change"],
    mode="lines", name="Daily Change",
    line=dict(color="#636EFA", width=0.7),
))

fig.add_trace(go.Scatter(
    x=extreme_jumps["date"], y=extreme_jumps["daily_change"],
    mode="markers", name=f"Extreme (|change| > {threshold:,.0f} MW)",
    marker=dict(color="#EF553B", size=6, symbol="x"),
))

fig.add_hline(y=threshold, line_dash="dash", line_color="yellow", opacity=0.5)
fig.add_hline(y=-threshold, line_dash="dash", line_color="yellow", opacity=0.5)

fig.update_layout(
    title="Day-over-Day Change in RT Daily Avg Load",
    xaxis_title="Date",
    yaxis_title="Change (MW)",
    height=400,
    template="plotly_dark",
)
fig.show()

print(f"Extreme jumps (> 3 sigma): {len(extreme_jumps)}")
if len(extreme_jumps) > 0:
    print(extreme_jumps[["date", "rt_daily_avg", "daily_change"]].head(10).to_string(index=False))

## 6. DA vs RT Comparison

Scatter plot of DA vs RT daily average load over the overlap period. Pearson correlation. Monthly boxplot of DA-RT difference.

In [ ]:
# Merge DA and RT on overlap period
merged = pd.merge(rt_daily, da_daily, on="date", how="inner")
print(f"Overlap period: {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"Overlap days: {len(merged):,}")

corr = merged["rt_daily_avg"].corr(merged["da_daily_avg"])
print(f"Pearson correlation (DA vs RT daily avg): {corr:.4f}")

# Scatter plot: DA vs RT daily avg
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=merged["rt_daily_avg"], y=merged["da_daily_avg"],
    mode="markers", name="DA vs RT",
    marker=dict(color="#636EFA", size=3, opacity=0.5),
))

# Add 45-degree reference line
xy_min = min(merged["rt_daily_avg"].min(), merged["da_daily_avg"].min())
xy_max = max(merged["rt_daily_avg"].max(), merged["da_daily_avg"].max())
fig.add_trace(go.Scatter(
    x=[xy_min, xy_max], y=[xy_min, xy_max],
    mode="lines", name="1:1 Line",
    line=dict(color="yellow", dash="dash", width=1),
))

fig.update_layout(
    title=f"DA vs RT Daily Avg Load (corr={corr:.3f})",
    xaxis_title="RT Daily Avg (MW)",
    yaxis_title="DA Daily Avg (MW)",
    height=450,
    template="plotly_dark",
)
fig.show()

In [ ]:
# Monthly DA-RT difference boxplot
merged["da_rt_diff"] = merged["da_daily_avg"] - merged["rt_daily_avg"]
merged["month_name"] = merged["date"].dt.strftime("%b")
merged["month_num"] = merged["date"].dt.month

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig = px.box(
    merged,
    x="month_name",
    y="da_rt_diff",
    category_orders={"month_name": month_order},
    title="Monthly DA-RT Daily Avg Load Difference",
    labels={"da_rt_diff": "DA - RT (MW)", "month_name": "Month"},
    template="plotly_dark",
)
fig.add_hline(y=0, line_dash="dash", line_color="yellow", opacity=0.5)
fig.update_layout(height=400)
fig.show()

print(f"DA-RT difference stats:")
print(merged["da_rt_diff"].describe().to_string())

## 7. Diurnal Profile

Average hourly load shape by season. Weekday vs weekend overlay.

In [ ]:
# Assign season
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df_rt_profile = df_rt.copy()
df_rt_profile["month"] = df_rt_profile["date"].dt.month
df_rt_profile["season"] = df_rt_profile["month"].apply(assign_season)

# Average hourly load by season
seasonal_profile = (
    df_rt_profile.groupby(["season", "hour_ending"])["rt_load_mw"]
    .mean()
    .reset_index()
)

season_order = ["Winter", "Spring", "Summer", "Fall"]

fig = px.line(
    seasonal_profile,
    x="hour_ending",
    y="rt_load_mw",
    color="season",
    category_orders={"season": season_order},
    title="Average Hourly RT Load Shape by Season",
    labels={"rt_load_mw": "Avg RT Load (MW)", "hour_ending": "Hour Ending"},
    template="plotly_dark",
)
fig.update_layout(height=450)
fig.update_xaxes(dtick=1)
fig.show()

In [ ]:
# Weekday vs weekend hourly profile
df_rt_profile["dow"] = df_rt_profile["date"].dt.dayofweek
df_rt_profile["day_type"] = df_rt_profile["dow"].apply(lambda x: "Weekend" if x >= 5 else "Weekday")

daytype_profile = (
    df_rt_profile.groupby(["day_type", "hour_ending"])["rt_load_mw"]
    .mean()
    .reset_index()
)

fig = px.line(
    daytype_profile,
    x="hour_ending",
    y="rt_load_mw",
    color="day_type",
    title="Average Hourly RT Load — Weekday vs Weekend",
    labels={"rt_load_mw": "Avg RT Load (MW)", "hour_ending": "Hour Ending", "day_type": "Day Type"},
    template="plotly_dark",
    color_discrete_map={"Weekday": "#636EFA", "Weekend": "#EF553B"},
)
fig.update_layout(height=450)
fig.update_xaxes(dtick=1)
fig.show()

## 8. Feature Sanity Check

Run `load_features.build(df_da_load, df_rt_load)`. Verify shape, columns. Check that `load_peak_ratio = peak / avg`. Correlation heatmap of all load features.

In [ ]:
from src.like_day_forecast.features import load_features

# Build features using both DA and RT load
df_features = load_features.build(df_da_load=df_da, df_rt_load=df_rt)

print(f"Feature DataFrame shape: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")
print(f"Date range: {df_features['date'].min()} to {df_features['date'].max()}")
print()
print("Feature statistics:")
print(df_features.describe().round(2).to_string())

In [ ]:
# Verify load_peak_ratio = load_daily_peak / load_daily_avg
expected_ratio = df_features["load_daily_peak"] / df_features["load_daily_avg"]
ratio_match = np.allclose(
    df_features["load_peak_ratio"].dropna(),
    expected_ratio.loc[df_features["load_peak_ratio"].dropna().index],
    rtol=1e-6,
)
print(f"load_peak_ratio = peak / avg formula verified: {ratio_match}")

if not ratio_match:
    diff = (df_features["load_peak_ratio"] - expected_ratio).abs()
    print(f"  Max absolute difference: {diff.max():.6f}")
    print(f"  Rows with mismatch: {(diff > 1e-6).sum()}")

In [ ]:
# Correlation heatmap of load features
feature_cols = [c for c in df_features.columns if c != "date"]
corr_matrix = df_features[feature_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Load Feature Correlation Heatmap",
    template="plotly_dark",
)
fig.update_layout(height=550, width=700)
fig.show()

## 9. Recent Spot Check

Last 7 days summary table. Last 3 days hourly load profiles.

In [ ]:
# Last 7 days summary — RT metered load
last_7_dates = sorted(df_rt["date"].unique())[-7:]
df_rt_last7 = df_rt[df_rt["date"].isin(last_7_dates)]

summary_7d = df_rt_last7.groupby("date").agg(
    hours=("hour_ending", "nunique"),
    avg_mw=("rt_load_mw", "mean"),
    peak_mw=("rt_load_mw", "max"),
    valley_mw=("rt_load_mw", "min"),
).reset_index()
summary_7d["peak_ratio"] = (summary_7d["peak_mw"] / summary_7d["avg_mw"]).round(3)

print("Last 7 days — RT Metered Load Summary:")
print(summary_7d.to_string(index=False, float_format="{:,.0f}".format))

In [ ]:
# Last 3 days hourly load profiles
last_3_dates = sorted(df_rt["date"].unique())[-3:]
df_rt_last3 = df_rt[df_rt["date"].isin(last_3_dates)].copy()
df_rt_last3["date_str"] = df_rt_last3["date"].dt.strftime("%Y-%m-%d")

fig = px.line(
    df_rt_last3,
    x="hour_ending",
    y="rt_load_mw",
    color="date_str",
    title="Last 3 Days — Hourly RT Metered Load Profile",
    labels={"rt_load_mw": "RT Load (MW)", "hour_ending": "Hour Ending", "date_str": "Date"},
    template="plotly_dark",
)
fig.update_layout(height=400)
fig.update_xaxes(dtick=1)
fig.show()

## 10. Validation Summary

Automated pass/fail checks for load data quality.

In [ ]:
from datetime import date

checks = []

# 1. RT date range covers 2014
rt_min_year = df_rt["date"].min().year
checks.append(("RT date range covers 2014", rt_min_year <= 2014))

# 2. DA date range covers 2020
da_min_year = df_da["date"].min().year
checks.append(("DA date range covers 2020", da_min_year <= 2020))

# 3. All dates have 24 hours (excluding today)
today = pd.Timestamp(date.today())
for label, df in [("RT", df_rt), ("DA", df_da)]:
    df_excl_today = df[df["date"] < today]
    hours_per_day = df_excl_today.groupby("date")["hour_ending"].nunique()
    all_24 = (hours_per_day == 24).all()
    checks.append((f"{label} all dates have 24h (excl today)", bool(all_24)))

# 4. No duplicates
for label, df in [("RT", df_rt), ("DA", df_da)]:
    n_dupes = df.duplicated(subset=["date", "hour_ending"], keep=False).sum()
    checks.append((f"{label} no duplicates", n_dupes == 0))

# 5. Zero nulls in load columns
rt_nulls = df_rt["rt_load_mw"].isnull().sum()
da_nulls = df_da["da_load_mw"].isnull().sum()
checks.append(("RT zero nulls in rt_load_mw", rt_nulls == 0))
checks.append(("DA zero nulls in da_load_mw", da_nulls == 0))

# 6. Load in physical bounds (30,000 - 200,000 MW)
rt_in_bounds = (df_rt["rt_load_mw"] >= LOAD_LOW).all() and (df_rt["rt_load_mw"] <= LOAD_HIGH).all()
da_in_bounds = (df_da["da_load_mw"] >= LOAD_LOW).all() and (df_da["da_load_mw"] <= LOAD_HIGH).all()
checks.append(("RT load in physical bounds (30k-200k MW)", bool(rt_in_bounds)))
checks.append(("DA load in physical bounds (30k-200k MW)", bool(da_in_bounds)))

# 7. DA-RT correlation > 0.9
da_rt_corr = merged["rt_daily_avg"].corr(merged["da_daily_avg"])
checks.append((f"DA-RT correlation > 0.9 (actual: {da_rt_corr:.4f})", da_rt_corr > 0.9))

# 8. load_peak_ratio formula verified
expected_ratio = df_features["load_daily_peak"] / df_features["load_daily_avg"]
ratio_ok = np.allclose(
    df_features["load_peak_ratio"].dropna(),
    expected_ratio.loc[df_features["load_peak_ratio"].dropna().index],
    rtol=1e-6,
)
checks.append(("load_peak_ratio = peak / avg verified", bool(ratio_ok)))

# Print results
print("=" * 65)
print("VALIDATION SUMMARY")
print("=" * 65)
all_pass = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")
print("=" * 65)
print(f"\nOverall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")